In [1]:
# --- Imports & Constants--- #

import pandas as pd
import numpy as np
from pathlib import Path
from main import preprocess, encode, train, get_preds

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

DATA_PATH = Path("data")
TRAINING_PATH = DATA_PATH / "train.csv" 
TESTING_PATH = DATA_PATH / "test.csv"

In [10]:
# --- Load Data --- #

train_df = pd.read_csv(TRAINING_PATH)
test_df = pd.read_csv(TESTING_PATH)

display(train_df.head())

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
train_df = train_df.drop(columns=["Cabin", "Name", "Ticket", "PassengerId"])
test_df = test_df.drop(columns=["Cabin", "Name", "Ticket"])

train_df = train_df.dropna()
print(train_df.shape)


(712, 8)


In [4]:
# --- preprocess --- #
train_X_df, train_y_df = preprocess(train_df)
test_X_df, _ = preprocess(test_df)

In [5]:
from IPython import get_ipython
get_ipython().kernel.debugger

In [6]:
# --- encode --- #

print(train_X_df)
breakpoint()

train_X_encoded = encode(train_X_df)

test_X_encoded = encode(test_X_df)

     Pclass     Sex   Age  SibSp  Parch     Fare Embarked
0         3    male  22.0      1      0   7.2500        S
1         1  female  38.0      1      0  71.2833        C
2         3  female  26.0      0      0   7.9250        S
3         1  female  35.0      1      0  53.1000        S
4         3    male  35.0      0      0   8.0500        S
..      ...     ...   ...    ...    ...      ...      ...
885       3  female  39.0      0      5  29.1250        Q
886       2    male  27.0      0      0  13.0000        S
887       1  female  19.0      0      0  30.0000        S
889       1    male  26.0      0      0  30.0000        C
890       3    male  32.0      0      0   7.7500        Q

[712 rows x 7 columns]


/home/cahxree/dev/ml-grindsheet/titanic/main.py:55: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["Sex"] = df["Sex"].str.strip().str.lower().map({"male": 0, "female": 1})


In [7]:
print(train_X_encoded.shape)
print(train_y_df.shape)

(712, 8)
(712,)


In [8]:
print(test_X_encoded.isna().any())
print(test_X_encoded.shape)

test_X_encoded["Age"] = test_X_encoded["Age"].fillna(test_X_encoded["Age"].mean())
test_X_encoded["Fare"] = test_X_encoded["Fare"].fillna(test_X_encoded["Fare"].mean())
print(test_X_encoded)
print(test_X_encoded.isna().any())


Pclass         False
Sex            False
Age             True
SibSp          False
Parch          False
Fare            True
PassengerId    False
Embarked_Q     False
Embarked_S     False
dtype: bool
(418, 9)
     Pclass  Sex       Age  SibSp  Parch      Fare  PassengerId  Embarked_Q  \
0         3    0  34.50000      0      0    7.8292          892        True   
1         3    1  47.00000      1      0    7.0000          893       False   
2         2    0  62.00000      0      0    9.6875          894        True   
3         3    0  27.00000      0      0    8.6625          895       False   
4         3    1  22.00000      1      1   12.2875          896       False   
..      ...  ...       ...    ...    ...       ...          ...         ...   
413       3    0  30.27259      0      0    8.0500         1305       False   
414       1    1  39.00000      0      0  108.9000         1306       False   
415       3    0  38.50000      0      0    7.2500         1307       False   


In [9]:
models = {
    "Logistic": LogisticRegression(max_iter=1000),
    "SVM": SVC(),
    "DecisionTree": DecisionTreeClassifier()
}

for name, model in models.items():
    pipe = train((name, model), train_X_encoded, train_y_df)
    predictions = get_preds(name, pipe, test_X_encoded)
    predictions.to_csv(f"{name}_preds.csv", index=False)

Pipeline(steps=[('scaler', StandardScaler()),
                ('Logistic', LogisticRegression(max_iter=1000))])
Pipeline(steps=[('scaler', StandardScaler()), ('SVM', SVC())])
Pipeline(steps=[('scaler', StandardScaler()),
                ('DecisionTree', DecisionTreeClassifier())])


In [8]:
# --- Inspect & Explore Data --- #

# Look for things that would prevent us from training (i.e. Non-numerical features, missing data)
display(train_df.head())
print(train_df.isna().any())

train_df["Age"] = train_df["Age"].fillna(train_df["Age"].mean())
print(train_df.isna().any())

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


PassengerId    False
Survived       False
Pclass         False
Name           False
Sex            False
Age             True
SibSp          False
Parch          False
Ticket         False
Fare           False
Cabin           True
Embarked        True
dtype: bool
PassengerId    False
Survived       False
Pclass         False
Name           False
Sex            False
Age            False
SibSp          False
Parch          False
Ticket         False
Fare           False
Cabin           True
Embarked        True
dtype: bool


In [11]:
print(train_df["Embarked"].isna().sum())
print(train_df["Age"].isna().sum())

2
177


In [16]:
print(train_df[train_df["Age"].isna()])
print(train_df.shape)

     PassengerId  Survived  Pclass                                      Name  \
5              6         0       3                          Moran, Mr. James   
17            18         1       2              Williams, Mr. Charles Eugene   
19            20         1       3                   Masselmani, Mrs. Fatima   
26            27         0       3                   Emir, Mr. Farred Chehab   
28            29         1       3             O'Dwyer, Miss. Ellen "Nellie"   
..           ...       ...     ...                                       ...   
859          860         0       3                          Razi, Mr. Raihed   
863          864         0       3         Sage, Miss. Dorothy Edith "Dolly"   
868          869         0       3               van Melkebeke, Mr. Philemon   
878          879         0       3                        Laleff, Mr. Kristo   
888          889         0       3  Johnston, Miss. Catherine Helen "Carrie"   

        Sex  Age  SibSp  Parch      Tic

In [ ]:
# -- Clean & Preprocess Data --- #

In [ ]:
# --- Train Models --- #

In [ ]:
# --- Evaluate Models --- #